# BDIViz User Test Case 2

## 0. Load Dataset

In [1]:
import pandas as pd
import numpy as np

import panel as pn
import bdikit as bdi
from bdiviz import BDISchemaMatchingHeatMap

In [2]:
# Dou sample #1
# source = pd.read_csv("Dou-ucec-sample-1.csv").drop(columns=["Unnamed: 0"])

# Dou sample #2
source = pd.read_csv("Dou-ucec-sample-2.csv").drop(columns=["Unnamed: 0"])


target = pd.read_csv("Dou-ucec-confirmatory.csv")

In [3]:
source

,Country,Race,Myometrial_invasion_Specify,Progesterone_Receptor,MSH6,CIBERSORT_Mast _cells _resting,CNV_class,ESTIMATE_ImmuneScore,Path_Stage_Reg_Lymph_Nodes-pN,Path_Stage_Primary_Tumor-pT
0,United States,White,under 50 %,Cannot be determined,Loss of nuclear expression,0.000000,CNV_LOW,4885.608881,pN0,pT1a (FIGO IA)
1,United States,White,under 50 %,Cannot be determined,Intact nuclear expression,0.000000,CNV_LOW,3632.199987,pNX,pT1a (FIGO IA)
2,United States,White,under 50 %,Cannot be determined,Intact nuclear expression,0.000000,CNV_LOW,6602.912323,pN0,pT1a (FIGO IA)
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,United States,White,under 50 %,Cannot be determined,Intact nuclear expression,0.065982,CNV_LOW,4462.910274,pNX,pT1a (FIGO IA)
...,...,...,...,...,...,...,...,...,...,...
148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Visualization Interface

In [4]:
heatmap_manager = BDISchemaMatchingHeatMap(
    source,
    target=target,
    top_k=20,
    # additional_sources=additional_sources,
    ai_assitant=False,
)
heatmap_manager.plot_heatmap()

Extracting features from 10 columns...


  0%|          | 0/10 [00:00<?, ?it/s]

Extracting features from 214 columns...


  0%|          | 0/214 [00:00<?, ?it/s]

Unmatched latitude columns: ['xCell_T_cell_CD4+_(non-regulatory)', 'xCell_T_cell_regulatory_(Tregs)', 'Cibersort_T_cell_regulatory_(Tregs)']


Column(scroll=True)
    [0] Row(styles={'background': '...}, width=1200)
        [0] Select(name='Source Column', options=['Country', 'Race', ...], value='Country', width=120)
        [1] Select(name='Candidate Type', options=['All', 'enum', ...], value='All', width=120)
        [2] IntSlider(end=5, name='Similar Sources', width=150)
        [3] FloatSlider(name='Candidate Threshold', step=0.01, value=0.1, width=150)
        [4] Column
            [0] Button(button_type='success', name='Accept Match')
            [1] Button(button_type='danger', name='Reject Match')
            [2] Button(button_type='warning', name='Discard Column')
        [5] Column
            [0] Button(button_style='outline', button_type='warning', name='Undo')
            [1] Button(button_style='outline', button_type='primary', name='Redo')
        [6] Column
            [0] Button(button_type='primary', name='Show/Hide AI Assistant')
            [1] Button(button_type='primary', name='Show/Hide Operation Log')
    [1] ParamFunction(function, _pane=Str, defer_load=False)
    [2] Spacer(height=5)
    [3] Column
        [0] ParamFunction(function, _pane=Column, defer_load=False)

## 2. Schema Matching with BDIViz Results

In [5]:
from bdikit.mapping_algorithms.column_mapping.algorithms import TwoPhaseSchemaMatcher

two_phase_viz = TwoPhaseSchemaMatcher(top_k_matcher=heatmap_manager)
column_mappings = bdi.match_schema(source, target=target, method=two_phase_viz)
column_mappings

,source,target
0,Country,Participant_country
1,Race,Race
2,Myometrial_invasion_Specify,Myometrial_invasion_present_specify
3,Progesterone_Receptor,Ancillary_studies_progesterone_receptor
4,MSH6,Ancillary_studies_msh6
5,CIBERSORT_Mast _cells _resting,Cibersort_Mast_cell_resting
6,CNV_class,CNV_status
7,ESTIMATE_ImmuneScore,Estimate_ImmuneScore
8,Path_Stage_Reg_Lymph_Nodes-pN,Pathologic_staging_regional_lymph_nodes_pn
9,Path_Stage_Primary_Tumor-pT,Pathologic_staging_primary_tumor_pt


## 3. Value Matchings

In [6]:
column_mappings = column_mappings[column_mappings["target"].str.strip().astype(bool)]
mappings = bdi.match_values(
    source,
    column_mapping=column_mappings,
    target=target,
    method="tfidf",
)

for mapping in mappings:
    print(f"{mapping.attrs['source']} => {mapping.attrs['target']}")
    display(mapping)
    print("")

Country => Participant_country


,source,target,similarity
0,United States,United States,1.0
1,NaN,NaN,1.0
2,Ukraine,Ukraine,1.0
3,Poland,Poland,1.0
4,Other_specify,NaN,NaN



Race => Race


,source,target,similarity
0,White,White,1.0
1,NaN,NaN,1.0
2,Asian,Asian,1.0
3,Not Reported,Not Reported,1.0
4,Black or African American,Black or African American,1.0



Myometrial_invasion_Specify => Myometrial_invasion_present_specify


...

,source,target,similarity
0,NaN,NaN,1.000
1,CNV_LOW,CNV_L,0.631
2,CNV_HIGH,CNV_H,0.623



Path_Stage_Reg_Lymph_Nodes-pN => Pathologic_staging_regional_lymph_nodes_pn


,source,target,similarity
0,pN0,pN0,1.0
1,pNX,pNX,1.0
2,NaN,NaN,1.0
3,pN2 (FIGO IIIC2),pN2 (FIGO IIIC2),1.0
4,pN1 (FIGO IIIC1),pN1 (FIGO IIIC1),1.0



Path_Stage_Primary_Tumor-pT => Pathologic_staging_primary_tumor_pt


,source,target,similarity
0,pT1a (FIGO IA),pT1a (FIGO IA),1.0
1,NaN,NaN,1.0
2,pT3a (FIGO IIIA),pT3a (FIGO IIIA),1.0
3,pT1 (FIGO I),pT1 (FIGO I),1.0
4,pT1b (FIGO IB),pT1b (FIGO IB),1.0
5,pT2 (FIGO II),pT2 (FIGO II),1.0
6,pT3b (FIGO IIIB),pT3b (FIGO IIIB),1.0
